# Validation Test Evaluation - Droplet spreading under gravity

This notebook and the corresponding simulation setup notebook (DropletSpreadingUnderGravity_Run.ipynb) are part of published results (cf. 5.2) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("CLpaper_DropletSpreadingUnderGravity");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\CLpaper_DropletSpreadingUnderGravity");

Opening existing database '\\fdygitrunner\ValidationTests\databases\CLpaper_DropletSpreadingUnderGravity'.


In [3]:
using System.IO;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
static var sessions = wmg.Sessions;
//static var sessions = databases.Pick(0).Sessions;
sessions

#0: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo5_AMR2_Restart1	12/11/2025 12:02:12 PM	e5e138c7...
#1: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo5	12/6/2025 11:06:15 PM	10c6c098...
#2: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo2_AMR2	11/27/2025 3:50:12 PM	1c470f8b...
#3: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo0.01_AMR2	11/27/2025 3:50:01 PM	c95aa2e0...
#4: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo2	11/27/2025 3:49:39 PM	63888cc3...
#5: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo1	11/27/2025 3:49:28 PM	20cb8396...
#6: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo0.1	11/27/2025 3:49:17 PM	08dc6d76...
#7: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo0.01	11/27/2025 3:49:07 PM	bf3560db...
#8: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo5_AMR2*	11/27/2025 3:50:23 PM	6c1a0b25...


In [5]:
string[] EoStudy = { "0.01", "0.1", "1", "2", "5" };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_DropletSpreadingUnderGravity");

In [6]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (string Eo in EoStudy) {
    string studyName = $"StaticDropletOnSlipWall_Eo{Eo}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName) && !sess.Name.Contains("_AMR2"));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

Searching for: StaticDropletOnSlipWall_Eo0.01
found
Searching for: StaticDropletOnSlipWall_Eo0.1
found
Searching for: StaticDropletOnSlipWall_Eo1
found
Searching for: StaticDropletOnSlipWall_Eo2
found
Searching for: StaticDropletOnSlipWall_Eo5
found


#0: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo0.01	11/27/2025 3:49:07 PM	bf3560db...
#1: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo0.1	11/27/2025 3:49:17 PM	08dc6d76...
#2: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo1	11/27/2025 3:49:28 PM	20cb8396...
#3: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo2	11/27/2025 3:49:39 PM	63888cc3...
#4: CLpaper_DropletSpreadingUnderGravity	StaticDropletOnSlipWall_Eo5	12/6/2025 11:06:15 PM	10c6c098...


In [8]:
NUnit.Framework.Assert.AreEqual(5, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [77]:
bool calcComputeTimes = false;

In [78]:
if (calcComputeTimes) {

foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

### Simulation properties for evaluation

In [9]:
static double r0 = 0.01;    // initial drop radius
static double statCA = 50 * (Math.PI / 180);  // static contact angle in rad
static double e0 = r0 * (1.0 - Math.Cos(statCA))*Math.Sqrt(Math.PI / (2.0*(statCA - Math.Sin(statCA)*Math.Cos(statCA))));

static double[] EoS = new double[] { 0.01, 0.1, 1, 2, 5};

## FIGURE 7 - Normalized thickness e*

In [10]:
public static (double[] x, double[] y) GetTerminalInterfaceShape_XYvalues(ISessionInfo evalSess) {

    var terminalStep = evalSess.Timesteps.WithoutSubSteps().OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
    //Console.WriteLine($"Terminal time step: {terminalStep.PhysicalTime}");
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(terminalStep.PhysicalTime);

    MultidimensionalArray interP = BoSSS.Solution.XNSECommon.XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet:10);
    double[] x = interP.ExtractSubArrayShallow(new int[] { -1, 0 }).To1DArray();
    double[] y = interP.ExtractSubArrayShallow(new int[] { -1, 1 }).To1DArray();

    return (x, y);
    
}

In [51]:
public static (double[] time, double[] values) LoadReferenceData(string dat) {

    string[] lines = File.ReadAllLines(dat);
    double[] time = new double[lines.Length];
    double[] valueDat = new double[lines.Length];
    for (int i = 0; i < lines.Length; i++) {
        time[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
        valueDat[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[1]);
    }

    return (time, valueDat);
}

In [60]:
public static Plot2Ddata GetNormalizedThickness_Plot2Ddata(List<ISessionInfo> dataSess) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Eo";
    plt.Ylabel = "e*";

    double[] eAst = new double[dataSess.Count()];
    for (int i = 0; i < EoS.Length; i++) {
        var shape = GetTerminalInterfaceShape_XYvalues(dataSess.ElementAt(i));
        //Console.WriteLine($"terminal drop thickness = {shape.y.Max()}");
        eAst[i] = shape.y.Max() / e0;
    }

    // BoSSS
    var fmt = new PlotFormat();
    fmt.Style = Styles.Points;
    fmt.PointType  = PointTypes.Times;
    //fmt.PointSize = 0.5;
    fmt.LineColor = LineColors.Black;

    plt.AddDataGroup("BoSSS", EoS, eAst, fmt);

    // Dupont & Legendre
    (double[] time, double[] valueDat) = LoadReferenceData("DropUnderGravPlot_Dupont.csv");
    fmt = new PlotFormat();
    fmt.Style = Styles.Points;
    fmt.PointType  = PointTypes.OpenCircle;
    fmt.LineColor = LineColors.Black;
    plt.AddDataGroup("Dupont & Legendre", time, valueDat, fmt);  

    // Gründing
    (time, valueDat) = LoadReferenceData("DropUnderGravPlot_ALE.csv");
    fmt = new PlotFormat();
    fmt.Style = Styles.Points;
    fmt.PointType  = PointTypes.Plus;
    fmt.LineColor = LineColors.Black;
    plt.AddDataGroup("Gründing", time, valueDat, fmt);  


    // asymptotic solution E0 << 1
    fmt = new PlotFormat();
    fmt.Style = Styles.Lines;
    fmt.DashType = DashTypes.Dashed;
    //fmt.PointSize = 0.5;
    fmt.LineColor = LineColors.Black;
    plt.AddDataGroup("", new double[] {0, 10}, new double[] { 1.0, 1.0}, fmt);

    // asymptotic solution E0 >> 1
    fmt = new PlotFormat();
    fmt.Style = Styles.Lines;
    fmt.DashType = DashTypes.Dashed;
    //fmt.PointSize = 0.5;
    fmt.LineColor = LineColors.Black;
    double[] xEo = GenericBlas.Linspace(0.5, 10, 100);
    double[] yEinfty = new double[xEo.Length];
    for (int i = 0; i < xEo.Length; i++) {
        yEinfty[i] = 2.0 * (r0 / Math.Sqrt(xEo[i]) * Math.Sin(statCA / 2.0)) / e0;
    }
    plt.AddDataGroup("", xEo, yEinfty, fmt);

    // exact solution
    (time, valueDat) = LoadReferenceData("DropUnderGravPlot_exact.csv");
    fmt = new PlotFormat();
    fmt.Style = Styles.Lines;
    fmt.LineColor = LineColors.Black;
    plt.AddDataGroup("", time, valueDat, fmt);  


    plt.LogX = true;
    plt.XrangeMin = 0.005;
    plt.XrangeMax = 10.0;
    plt.YrangeMin = 0.4;
    plt.YrangeMax = 1.2;
    plt.ShowLegend = true;
    plt.LegendAlignment = new string[] { "i", "b", "l"};

    return plt;
}

In [61]:
Plot2Ddata thickPlt = GetNormalizedThickness_Plot2Ddata(evalSess);

thickPlt.ToGnuplot().PlotSVG(xRes:600,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 1.2 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 e* 
 
 
 
 
 Eo 
 
 
 
 
 BoSSS 
 
 
 BoSSS 
 
 
 
 
 
 
 
 
 
 
 
 Dupont Legendre 
 
 
 Dupont Legendre 
 
 
 
 
 
 
 
 
 
 
 
 
 Gründing 
 
 
 Gründing 
 
 
 
 
 
 
 
 
 
 
 
 gnuplot_plot_4 
 
 
 
 gnuplot_plot_5 
 
	<path stroke='rgb( 0, 0, 0)' stroke-dasharray='2.5,4.0' d='M401.5,36.1 L404.0,48.2 L409.8,75.5 L415.2,99.3 L420.2,120.4 L424.7,139.2 L429.0,156.1 L433.0,171.5
 L436.8,185.5 L440.4,198.3 L443.7,210.1 L446.9,221.0 L449.9,231.1 L452.8,240.5 L455.6,249.4 L458.3,257.6
 L460.8,265.4 L463.3,272.7 L465.6,279.6 L467.9,286.2 L470.1,292.4 L472.2,298.3 L474.2,303.9 L476.2,309.2
 L478.2,314.3 L480.0,319.2 L481.8,323.9 L483.6,328.4 L485.3,332.6 L487.0,336.8 L488.6,340.7 L490.2,344.5
 L491.7,348.2 L493.2,351.8 L494.7,355.2 L496.2,358.5 L497.6,361.7 L498.9,364.8 L500.3,367.8 L501.6,370.7
 L502.9,373.5 L504.2,376.2 L505.4,378.8 L506.6,381.4 L507.8,383.9 L509.0,386.3 L510.1,388.7 L511.3,391.0
 L512.4,393.2 L513.5,395.4 L514.5,397.5 L515.6,399.6 L516.6,401.6 L517.6,403.6 L518.6,405.5 L519.6,407.3
 L520.6,409.2 L521.6,411.0 L522.5,412.7 L523.4,414.4 L524.4,416.1 L525.3,417.8 L526.1,419.4 L527.0,420.9
 L527.9,422.5 L528.7,424.0 L529.6,425.5 L530.4,426.9 L531.2,428.3 L532.0,429.7 L532.8,431.1 L533.6,432.4
 L534.4,433.7 L535.2,435.0 L535.9,436.3 L536.7,437.6 L537.4,438.8 L538.2,440.0 L538.9,441.2 L539.6,442.3
 L539.7,442.4 '/> 
 
 gnuplot_plot_6

In [ ]:
double[] exactValues = new double[EoS.Length];
Plot2Ddata.XYvalues exactSol = thickPlt.dataGroups[5];
for (int i = 0; i < EoS.Length; i++) {
    // get exact value from file
    double value = 0.0;
    double minDist = double.MaxValue;
    for (int j = 0; j < exactSol.Values.Length; j++) {
        double dist = Math.Abs(exactSol.Abscissas[j] - EoS[i]);
        if (dist < minDist) {
            value = exactSol.Values[j];
            minDist = dist;
        }
    }
    exactValues[i] = value;
}

In [33]:
exactValues

[ 0.99887, 0.98667, 0.84739, 0.72462, 0.53293 ]

In [ ]:
double[] tolerances = new double[] { 0.01, 0.01, 0.02, 0.01, 0.01 };
for (int i = 0; i < EoS.Length; i++) {
    NUnit.Framework.Assert.AreEqual(exactValues[i], thickPlt.dataGroups[0].Values[i], tolerances[i], 
        $"Eo = {EoS[i]}: normalized thickness deviates too much from expected value {exactValues[i]}");
}   

## FIGURE 8 - Terminal shapes

In [62]:
public static Plot2Ddata GetTerminalShapes_Plot2Ddata(List<ISessionInfo> dataSess, string[] study) {

    Plot2Ddata datShape =  new Plot2Ddata();

    int lcIdx = 1;
    for(int i = 0; i < study.Length; i++) {
        foreach (var dataS in dataSess) {
            if (dataS.Name.Contains(study[i])) {
                var shape = GetTerminalInterfaceShape_XYvalues(dataS);

                var fmt = new PlotFormat();
                fmt.Style = Styles.Lines;
                fmt.Style      = Styles.Points;
                fmt.PointType  = PointTypes.Dot;
                fmt.LineColor = (LineColors)(lcIdx);

                datShape.AddDataGroup((study[i]).Replace("_", "-"), shape.x, shape.y, fmt);
                lcIdx++;
            }
        }
    }

    datShape.XrangeMin = -0.03;
    datShape.XrangeMax = 0.03;
    datShape.YrangeMin = 0.0;
    datShape.YrangeMax = .01;

    datShape.ShowLegend = true;
    datShape.LegendAlignment = new string[] { "i", "t", "r"};

    return datShape;
}

In [63]:
Plot2Ddata shapePlt = GetTerminalShapes_Plot2Ddata(evalSess, new string[] { "Eo0.01", "Eo2", "Eo5" });

shapePlt.ToGnuplot().PlotSVG(xRes:1200,yRes:300)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.002 
 
 
 
 
 0.004 
 
 
 
 
 0.006 
 
 
 
 
 0.008 
 
 
 
 
 0.01 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Eo0.01 
 
 
 Eo0.01 
 
 
 
 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

In [71]:
double[] expectedLengths = new double[] { 0.031, 0.039, 0.049 };

In [76]:
for (int g = 0; g < shapePlt.dataGroups.Length; g++) {

    var grp = shapePlt.dataGroups[g];
    double clpLeft = 0.0;
    double clpRight = 0.0;
    for (int i = 0; i < grp.Abscissas.Length; i++) {
        if (grp.Abscissas[i] < clpLeft)
                clpLeft = grp.Abscissas[i];       
        if (grp.Abscissas[i] > clpRight)
                clpRight = grp.Abscissas[i];
    }
    double wettingLength = -clpLeft + clpRight;
    Console.WriteLine($"Wetting length for {grp.Name} = {wettingLength}");

    NUnit.Framework.Assert.AreEqual(expectedLengths[g], wettingLength, 0.001, 
        $"Wetting length for {grp.Name} deviates too much from expected value 0.0");
}

Wetting length for Eo0.01 = 0.031047482140975566
Wetting length for Eo2 = 0.03929655230010719
Wetting length for Eo5 = 0.04907248247569267
